In [1]:
import numpy as np
import pandas as pd
from rpy2.robjects import r, conversion, pandas2ri
from helper_functions import normalize_household_data, dict_to_named_list

test file must lie in the sampling Bias folder

In [2]:
pandas2ri.activate()
r.source('Simulator/Simulator.R')
model_r = r['simulate_and_reformat']

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


R[write to console]: 
Attaching package: ‘actuar’


R[write to console]: The following objects are masked from ‘package:stats’:

    sd, var


R[write to console]: The following object is masked from ‘package:grDevices’:

    cm




In [3]:
PARAM_NAMES = ['alpha', 'beta', 'delta',
               'mu_inf_SC', 'mu_inf_SA', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
               'mu_susc_C', 'mu_susc_A',
               'mu_protect_acq', 'mu_protect_transm']

PROCEDURES = ['pedcov', 'random', 'original_pedcov', 'original_random', 'adult', 'sampling1', 'samplingIG']

In [4]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.random.normal(0, 1, (batch_size, len(PARAM_NAMES)))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    return param_batch

test = prior(1).flatten()

In [5]:
def simulator(log_params: np.ndarray,
              selection_procedure: str,
              variant: str,
              minimal_length: int = 9,
              fixed_parameters_dict: dict = None) -> np.ndarray:
    """
    Simulate data with given parameters and reformat it to a numpy array.
    :param log_params: parameters for the simulation
    :param selection_procedure: selection procedure for the simulation (pedcov or random)
    :param variant: variant of the simulation (alpha or omicron)
    :param minimal_length: minimal length of the data
    :param fixed_parameters_dict: dictionary with fixed parameters for each variant
    :return: simulated data as numpy array
    """
    if variant not in ['alpha', 'omicron']:
        raise ValueError(f"Variant '{variant}' not supported. Must be 'alpha' or 'omicron'.")
    if selection_procedure not in PROCEDURES:
        raise ValueError(f"Selection procedure '{selection_procedure}' not supported. "
                         f"Must be one of {PROCEDURES}.")

    if fixed_parameters_dict is None:
        fixed_parameters = []
    else:
        fixed_parameters = fixed_parameters_dict[variant]

    # transform parameters to correct scale
    un_scaled_params = np.copy(log_params)  # copy to avoid changing input
    # create dict from param_names, params might have different length
    par_dict = {}
    p_i = 0
    for name in PARAM_NAMES:
        if name in fixed_parameters:
            par_dict.update({name: fixed_parameters[name]})
        # all parameters besides alpha and delta are log-transformed
        elif name == 'delta' or name == 'alpha':
            par_dict.update({name: un_scaled_params[p_i]})
            p_i += 1
        else:
            par_dict.update({name: np.exp(un_scaled_params[p_i])})
            p_i += 1
    # update dict with fixed hyperparameters, make sure these are strings
    par_dict.update({'variant': str(variant), 'selection_procedure': str(selection_procedure)})
    print(par_dict)

    # simulate data
    sim_data_r = model_r(dict_to_named_list(par_dict))
    # convert to pandas dataframe
    sim_data_full = conversion.rpy2py(sim_data_r)
    # normalize data and return as numpy array
    sim_data_norm = normalize_household_data(sim_data_full, minimal_length=minimal_length)
    return sim_data_norm

In [6]:
sim_data = simulator(test, selection_procedure='pedcov', variant='alpha')

{'alpha': 0.03448235512348866, 'beta': 0.4611152800730058, 'delta': 0.9828173356246526, 'mu_inf_SC': 2.290202526714815, 'mu_inf_SA': 2.6997660988244196, 'mu_inf_AI': 0.6721884563559837, 'mu_inf_AC': 0.39573290489404234, 'mu_inf_AA': 0.49028015322478785, 'mu_susc_C': 0.8213262829101918, 'mu_susc_A': 0.21253477638972051, 'mu_protect_acq': 1.6969831174241896, 'mu_protect_transm': 0.6745610282478005, 'variant': 'alpha', 'selection_procedure': 'pedcov'}
[1] "Simulation"


R[write to console]: Error in if (u < proba_inf) { : missing value where TRUE/FALSE needed



RRuntimeError: Error in if (u < proba_inf) { : missing value where TRUE/FALSE needed


In [ ]:
sim_data_both = simulator_both_variants(np.concatenate((test, test)), selection_procedure='pedcov')

In [ ]:
sim_data_both

In [ ]:
normalize_household_data(sim_data, minimal_length=9)

In [ ]:
sim_data[sim_data['id_hh_origin'] == "42"]

In [ ]:
sim_data['id_hh_origin'].unique()

In [ ]:
# extract the original household ids
sim_data['id_hh_origin'].nunique()

In [ ]:
# columns: date_sympt_norm, infect_status_norm, age_norm, protected
# rows: empty individuals in the beginning (households are of same size)
# last row: end_followup_norm with 1

In [ ]:
# load a presimulation using pickle
import pickle
with open('presimulations/presim_file_1.pkl', 'rb') as f:
    presim = pickle.load(f)
    
with open('presimulations/presim_file_2.pkl', 'rb') as f:
    presim_2 = pickle.load(f)

In [ ]:
presim[0]['prior_draws']#-presim_2[5]['prior_draws']

In [ ]:
len(presim)

In [ ]:
presim_2[0]['prior_draws']#[:, :, :, 0]*1000

In [ ]:
with open('valid_data.pickle', 'rb') as f:
    valid_data = pickle.load(f)

In [ ]:
valid_data['prior_draws']